# Library and my ip host

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, lit, schema_of_json
# from pyspark.sql.types import StructType, StructField, DateType, StringType, FloatType
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc, avg, min, max, greatest

In [2]:
spark_session = SparkSession.builder.appName("Average_calculation").remote("sc://spark-test:15002").getOrCreate()

In [3]:
import socket

hostname = socket.gethostname()
ip_address = socket.gethostbyname(hostname)

print(f"Tên máy: {hostname}")
print(f"Địa chỉ IP: {ip_address}")


Tên máy: DELL_VOSTRO14
Địa chỉ IP: 192.168.1.100


# ETL

In [4]:
df = spark_session.read \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "kafka-test:9092") \
        .option("subscribe", "air_quality_data") \
        .option("startingOffsets", "earliest") \
        .option("endingOffsets", "latest") \
        .load()
df.show()

+----+--------------------+----------------+---------+------+--------------------+-------------+
| key|               value|           topic|partition|offset|           timestamp|timestampType|
+----+--------------------+----------------+---------+------+--------------------+-------------+
|NULL|[7B 22 64 69 73 7...|air_quality_data|        5|     0|2026-04-08 09:34:...|            0|
|NULL|[7B 22 64 69 73 7...|air_quality_data|        5|     1|2026-04-08 09:34:...|            0|
|NULL|[7B 22 64 69 73 7...|air_quality_data|        5|     2|2026-04-08 09:34:...|            0|
|NULL|[7B 22 64 69 73 7...|air_quality_data|        5|     3|2026-04-08 09:34:...|            0|
|NULL|[7B 22 64 69 73 7...|air_quality_data|        5|     4|2026-04-08 09:34:...|            0|
|NULL|[7B 22 64 69 73 7...|air_quality_data|        5|     5|2026-04-08 09:34:...|            0|
|NULL|[7B 22 64 69 73 7...|air_quality_data|        5|     6|2026-04-08 09:34:...|            0|
|NULL|[7B 22 64 69 73 7...|air

In [5]:
object_air_data = df.withColumns(
    {
        "key": df["key"].cast("string"),
        "value": df["value"].cast("string")
    }
).select("value")

object_air_data.show()

+--------------------+
|               value|
+--------------------+
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
+--------------------+
only showing top 20 rows


In [6]:
import json

rows = object_air_data.select("value").limit(10).collect()
schema = schema_of_json(lit(json.dumps(json.loads(rows[9].asDict()["value"]))))
schema

Column<'schema_of_json({"district": "Ba Dinh", "city": "Ha Noi", "time": "2022-08-06T14:00", "pm10 (\u03bcg/m\u00b3)": 25.7, "pm2_5 (\u03bcg/m\u00b3)": 18.0, "carbon_monoxide (\u03bcg/m\u00b3)": 293.0, "carbon_dioxide (ppm)": NaN, "nitrogen_dioxide (\u03bcg/m\u00b3)": 14.6, "sulphur_dioxide (\u03bcg/m\u00b3)": 12.4, "ozone (\u03bcg/m\u00b3)": 71.0, "aerosol_optical_depth ()": 0.32, "dust (\u03bcg/m\u00b3)": 0.0, "uv_index ()": 2.15, "uv_index_clear_sky ()": 7.7})'>

In [7]:
df_parsed = object_air_data.withColumn("value", from_json(col("value"), schema))
df_parsed.show(5)

+--------------------+
|               value|
+--------------------+
|{NaN, NaN, NaN, H...|
|{NaN, NaN, NaN, H...|
|{0.43, NaN, 595.0...|
|{0.79, NaN, 438.0...|
|{0.42, NaN, 354.0...|
+--------------------+
only showing top 5 rows


In [8]:
analysis_df = df_parsed.select(
    "value.city",
    "value.district",
    "value.time",
    col("value.pm2_5 (μg/m³)").alias("pm25"),
    col("value.pm10 (μg/m³)").alias("pm10"),
    col("value.carbon_monoxide (μg/m³)").alias("co"),
    # col("value.carbon_dioxide (ppm)").alias("co2"),
    col("value.nitrogen_dioxide (μg/m³)").alias("no2"),
    col("value.sulphur_dioxide (μg/m³)").alias("so2"),
    col("value.ozone (μg/m³)").alias("o3"),
    col("value.aerosol_optical_depth ()").alias("aod"),
    col("value.dust (μg/m³)").alias("dust"),
    col("value.uv_index ()").alias("uv_idx"),
    col("value.uv_index_clear_sky ()").alias("uv_idx_clear_sky"),
).filter((col("pm25").isNaN() == False) & (col("pm10").isNaN() == False)).orderBy("value.time")

In [9]:
analysis_df.write.format("csv").mode("overwrite").option("header", "true").save("hdfs://hadoop-test:9000/data/csv/processed/")

KeyboardInterrupt: 

# Prepare data and save into sql database (datawarehouse) (postgresql)

In [ ]:
test_df = spark_session.read.format("csv").option("header", "true").load("hdfs://hadoop-test:9000/data/csv/processed/")

In [ ]:
def avarage_by(df, previous_rows_count, column_names):
    window = Window.orderBy("time").rowsBetween(1-previous_rows_count, 0)
    result = df
    for col_name in column_names:
        result = result.withColumn(f"avg_{col_name}_{previous_rows_count}h", avg(col_name).over(window))
    return result

In [ ]:
test_df = avarage_by(test_df, 24, ["pm25", "pm10"])
test_df = avarage_by(test_df, 1, ["o3"])
test_df = avarage_by(test_df, 8, ["o3"])
test_df = avarage_by(test_df, 1, ["co", "so2", "no2"])

In [ ]:
test_df.show(1000)

d:\apps\conda\envs\mmd_env\Lib\site-packages\pyspark\sql\connect\expressions.py:1091: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------+--------+----------------+-----+-----+------+-----+----+-----+----+----+------+----------------+------------------+------------------+---------+------------------+---------+----------+----------+
|  city|district|            time| pm25| pm10|    co|  no2| so2|   o3| aod|dust|uv_idx|uv_idx_clear_sky|      avg_pm25_24h|      avg_pm10_24h|avg_o3_1h|         avg_o3_8h|avg_co_1h|avg_so2_1h|avg_no2_1h|
+------+--------+----------------+-----+-----+------+-----+----+-----+----+----+------+----------------+------------------+------------------+---------+------------------+---------+----------+----------+
|Ha Noi| Ba Dinh|2022-08-04T07:00| 40.3| 58.0| 595.0| 29.7|16.8| 24.0|0.43| 0.0|  0.75|             0.8|              40.3|              58.0|     24.0|              24.0|    595.0|      16.8|      29.7|
|Ha Noi| Ba Dinh|2022-08-04T08:00| 30.0| 43.5| 552.0| 25.0|18.2| 49.0|0.52| 0.0|   2.1|            2.35|             35.15|             50.75|     49.0|              36.5|    552.0|   

In [ ]:
test_df.select("time", "city", "district", "avg_pm25_24h", "avg_pm10_24h", "avg_o3_1h", "avg_o3_8h", "avg_co_1h", "avg_so2_1h", "avg_no2_1h") \
        .write \
        .format("jdbc") \
        .option("url", f"jdbc:postgresql://{ip_address}:5432/air_quality_datawarehouse") \
        .option("dbtable", "average_values") \
        .option("user", "admin") \
        .option("password", "admin123") \
        .option("driver", "org.postgresql.Driver") \
        .option("batchsize", 1000) \
        .mode("overwrite") \
        .save()

d:\apps\conda\envs\mmd_env\Lib\site-packages\pyspark\sql\connect\expressions.py:1091: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


# Analysis

## Metric table

In [ ]:
import pandas as pd

bp_idx = [
    {"I": [0, 50, 100, 150, 200, 300, 400, 500]},
    {"BP_o3_1h": [0, 160, 200, 300, 400, 800, 1000, 1200]},
    {"BP_o3_8h": [0, 100, 120, 170, 210, 400]},
    {"BP_co": [0, 10000, 30000, 45000, 60000, 90000, 120000, 150000]},
    {"BP_so2": [0, 125, 350, 550, 800, 1600, 2100, 2630]},
    {"BP_no2": [0, 100, 200, 700, 1200, 2350, 3100, 3850]},
    {"BP_pm10": [0, 50, 150, 250, 350, 420, 500, 600]},
    {"BP_pm25": [0, 25, 50, 80, 150, 250, 350, 500]}
]
bp_dict = {list(d.keys())[0]: list(d.values())[0] for d in bp_idx}

df_bp = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in bp_dict.items()]))

df_bp = spark_session.createDataFrame(df_bp)

In [ ]:
df_bp.write.format("jdbc") \
        .option("url", f"jdbc:postgresql://{ip_address}:5432/air_quality_datawarehouse") \
        .option("dbtable", "bp_index") \
        .option("user", "admin") \
        .option("password", "admin123") \
        .option("driver", "org.postgresql.Driver") \
        .option("batchsize", 1000) \
        .mode("overwrite") \
        .save()

## AQI calculation

### load

In [ ]:
df_bp = spark_session.read.format("jdbc") \
        .option("url", f"jdbc:postgresql://{ip_address}:5432/air_quality_datawarehouse") \
        .option("dbtable", "bp_index") \
        .option("user", "admin") \
        .option("password", "admin123") \
        .option("driver", "org.postgresql.Driver") \
        .load()

df_bp = df_bp.orderBy(col("I"))

In [ ]:
bp_o3_1h_df = df_bp.select("I", "BP_o3_1h")
bp_o3_8h_df = df_bp.select("I", "BP_o3_8h")
bp_co_df = df_bp.select("I", "BP_co")
bp_so2_df = df_bp.select("I", "BP_so2")
bp_no2_df = df_bp.select("I", "BP_no2")
bp_pm10_df = df_bp.select("I", "BP_pm10")
bp_pm25_df = df_bp.select("I", "BP_pm25")

In [ ]:
average_df = spark_session.read.format("jdbc") \
        .option("url", f"jdbc:postgresql://{ip_address}:5432/air_quality_datawarehouse") \
        .option("dbtable", "average_values") \
        .option("user", "admin") \
        .option("password", "admin123") \
        .option("driver", "org.postgresql.Driver") \
        .load()

In [ ]:
avg_pm25_24h_df = average_df.select("time", "city", "district", "avg_pm25_24h")
avg_pm10_24h_df = average_df.select("time", "city", "district", "avg_pm10_24h")
avg_o3_1h_df = average_df.select("time", "city", "district", "avg_o3_1h")
avg_o3_8h_df = average_df.select("time", "city", "district", "avg_o3_8h")
avg_co_1h_df = average_df.select("time", "city", "district", "avg_co_1h")
avg_so2_1h_df = average_df.select("time", "city", "district", "avg_so2_1h")
avg_no2_1h_df = average_df.select("time", "city", "district", "avg_no2_1h")

### start calculating AQI

In [ ]:
min_dis_window = Window.partitionBy(col("time"), col("city"), col("district")).orderBy(col("dis"))

#### AQI calculation for PM2.5

In [ ]:
join_avg_pm25_24h_df = avg_pm25_24h_df.alias("avg_pm25_24h_df").join(bp_pm25_df.alias("bp_pm25_df"), col("avg_pm25_24h_df.avg_pm25_24h") >= col("bp_pm25_df.BP_pm25"), how="left")
join_avg_pm25_24h_df = join_avg_pm25_24h_df.join(bp_pm25_df.alias("bp_pm25_df2"), col("avg_pm25_24h_df.avg_pm25_24h") < col("bp_pm25_df2.BP_pm25"), how="left")

In [ ]:
join_avg_pm25_24h_df = join_avg_pm25_24h_df.withColumn("dis", col("bp_pm25_df2.BP_pm25") - col("bp_pm25_df.BP_pm25"))
join_avg_pm25_24h_df = join_avg_pm25_24h_df.withColumn("min_dis", min(col("dis")).over(min_dis_window)).filter(col("min_dis") == col("dis"))
join_avg_pm25_24h_df = join_avg_pm25_24h_df.drop(col("dis"), col("min_dis"))

In [ ]:
join_avg_pm25_24h_df = join_avg_pm25_24h_df.withColumn("aqi_pm25_24h", ((col("avg_pm25_24h_df.avg_pm25_24h") - col("bp_pm25_df.BP_pm25")) * (col("bp_pm25_df2.I") - col("bp_pm25_df.I")) / (col("bp_pm25_df2.BP_pm25") - col("bp_pm25_df.BP_pm25")) + col("bp_pm25_df.I")))

In [ ]:
join_avg_pm25_24h_df = join_avg_pm25_24h_df.withColumn("date", col("time").substr(1, 10))

In [ ]:
max_aqi_window = Window.partitionBy(col("date"), col("city"), col("district")).orderBy(col("avg_pm25_24h_df.avg_pm25_24h").desc())

join_avg_pm25_24h_df = join_avg_pm25_24h_df.withColumn("max_avg_pm25_24h", max(col("avg_pm25_24h_df.avg_pm25_24h")).over(max_aqi_window)) \
                    .filter(col("max_avg_pm25_24h") == col("avg_pm25_24h_df.avg_pm25_24h")) \
                    .drop(col("max_avg_pm25_24h"))

In [ ]:
join_avg_pm25_24h_df.show(2)

+----------------+------+--------+-----------------+---+-------+---+-------+-----------------+----------+
|            time|  city|district|     avg_pm25_24h|  I|BP_pm25|  I|BP_pm25|     aqi_pm25_24h|      date|
+----------------+------+--------+-----------------+---+-------+---+-------+-----------------+----------+
|2022-08-04T23:00|Ha Noi| Ba Dinh|46.64117647058825| 50|     25|100|     50| 93.2823529411765|2022-08-04|
|2022-08-05T00:00|Ha Noi| Ba Dinh|46.58333333333334| 50|     25|100|     50|93.16666666666669|2022-08-05|
+----------------+------+--------+-----------------+---+-------+---+-------+-----------------+----------+
only showing top 2 rows


#### AQI calculation for PM10

In [ ]:
join_avg_pm10_24h_df = avg_pm10_24h_df.alias("avg_pm10_24h_df").join(bp_pm10_df.alias("bp_pm10_df"), col("avg_pm10_24h_df.avg_pm10_24h") >= col("bp_pm10_df.BP_pm10"), how="left")
join_avg_pm10_24h_df = join_avg_pm10_24h_df.join(bp_pm10_df.alias("bp_pm10_df2"), col("avg_pm10_24h_df.avg_pm10_24h") < col("bp_pm10_df2.BP_pm10"), how="left")

In [ ]:
join_avg_pm10_24h_df = join_avg_pm10_24h_df.withColumn("dis", col("bp_pm10_df2.BP_pm10") - col("bp_pm10_df.BP_pm10"))
join_avg_pm10_24h_df = join_avg_pm10_24h_df.withColumn("min_dis", min(col("dis")).over(min_dis_window)).filter(col("min_dis") == col("dis"))
join_avg_pm10_24h_df = join_avg_pm10_24h_df.drop(col("dis"), col("min_dis"))

In [ ]:
join_avg_pm10_24h_df = join_avg_pm10_24h_df.withColumn("aqi_pm10_24h", ((col("avg_pm10_24h_df.avg_pm10_24h") - col("bp_pm10_df.BP_pm10")) * (col("bp_pm10_df2.I") - col("bp_pm10_df.I")) / (col("bp_pm10_df2.BP_pm10") - col("bp_pm10_df.BP_pm10")) + col("bp_pm10_df.I")))

In [ ]:
join_avg_pm10_24h_df = join_avg_pm10_24h_df.withColumn("date", col("time").substr(1, 10))

In [ ]:
max_aqi_window = Window.partitionBy(col("date"), col("city"), col("district")).orderBy(col("avg_pm10_24h_df.avg_pm10_24h").desc())

join_avg_pm10_24h_df = join_avg_pm10_24h_df.withColumn("max_avg_pm10_24h", max(col("avg_pm10_24h_df.avg_pm10_24h")).over(max_aqi_window)) \
                    .filter(col("max_avg_pm10_24h") == col("avg_pm10_24h_df.avg_pm10_24h")) \
                    .drop(col("max_avg_pm10_24h"))

In [ ]:
join_avg_pm10_24h_df.show(2)

+----------------+------+--------+-----------------+---+-------+---+-------+-----------------+----------+
|            time|  city|district|     avg_pm10_24h|  I|BP_pm10|  I|BP_pm10|     aqi_pm10_24h|      date|
+----------------+------+--------+-----------------+---+-------+---+-------+-----------------+----------+
|2022-08-04T23:00|Ha Noi| Ba Dinh|67.08235294117648| 50|     50|100|    150|58.54117647058824|2022-08-04|
|2022-08-05T00:00|Ha Noi| Ba Dinh|67.00555555555556| 50|     50|100|    150|58.50277777777778|2022-08-05|
+----------------+------+--------+-----------------+---+-------+---+-------+-----------------+----------+
only showing top 2 rows


#### AQI calculation for no2

In [ ]:
join_avg_no2_1h_df = avg_no2_1h_df.alias("avg_no2_1h_df").join(bp_no2_df.alias("bp_no2_df"), col("avg_no2_1h_df.avg_no2_1h") >= col("bp_no2_df.BP_no2"), how="left")
join_avg_no2_1h_df = join_avg_no2_1h_df.join(bp_no2_df.alias("bp_no2_df2"), col("avg_no2_1h_df.avg_no2_1h") < col("bp_no2_df2.BP_no2"), how="left")

In [ ]:
join_avg_no2_1h_df = join_avg_no2_1h_df.withColumn("dis", col("bp_no2_df2.BP_no2") - col("bp_no2_df.BP_no2"))
join_avg_no2_1h_df = join_avg_no2_1h_df.withColumn("min_dis", min(col("dis")).over(min_dis_window)).filter(col("min_dis") == col("dis"))
join_avg_no2_1h_df = join_avg_no2_1h_df.drop(col("dis"), col("min_dis"))

In [ ]:
join_avg_no2_1h_df = join_avg_no2_1h_df.withColumn("aqi_no2_1h", ((col("avg_no2_1h_df.avg_no2_1h") - col("bp_no2_df.BP_no2")) * (col("bp_no2_df2.I") - col("bp_no2_df.I")) / (col("bp_no2_df2.BP_no2") - col("bp_no2_df.BP_no2")) + col("bp_no2_df.I")))

In [ ]:
join_avg_no2_1h_df = join_avg_no2_1h_df.withColumn("date", col("time").substr(1, 10))

In [ ]:
max_aqi_window = Window.partitionBy(col("date"), col("city"), col("district")).orderBy(col("avg_no2_1h_df.avg_no2_1h").desc())

join_avg_no2_1h_df = join_avg_no2_1h_df.withColumn("max_avg_no2_1h", max(col("avg_no2_1h_df.avg_no2_1h")).over(max_aqi_window)) \
                    .filter(col("max_avg_no2_1h") == col("avg_no2_1h_df.avg_no2_1h")) \
                    .drop(col("max_avg_no2_1h"))

In [ ]:
join_avg_no2_1h_df.show(2)

+----------------+------+--------+----------+---+------+---+------+----------+----------+
|            time|  city|district|avg_no2_1h|  I|BP_no2|  I|BP_no2|aqi_no2_1h|      date|
+----------------+------+--------+----------+---+------+---+------+----------+----------+
|2022-08-04T19:00|Ha Noi| Ba Dinh|      78.9|  0|     0| 50|   100|     39.45|2022-08-04|
|2022-08-05T19:00|Ha Noi| Ba Dinh|      97.8|  0|     0| 50|   100|      48.9|2022-08-05|
+----------------+------+--------+----------+---+------+---+------+----------+----------+
only showing top 2 rows


#### AQI calculation for so2

In [ ]:
join_avg_so2_1h_df = avg_so2_1h_df.alias("avg_so2_1h_df").join(bp_so2_df.alias("bp_so2_df"), col("avg_so2_1h_df.avg_so2_1h") >= col("bp_so2_df.BP_so2"), how="left")
join_avg_so2_1h_df = join_avg_so2_1h_df.join(bp_so2_df.alias("bp_so2_df2"), col("avg_so2_1h_df.avg_so2_1h") < col("bp_so2_df2.BP_so2"), how="left")

In [ ]:
join_avg_so2_1h_df = join_avg_so2_1h_df.withColumn("dis", col("bp_so2_df2.BP_so2") - col("bp_so2_df.BP_so2"))
join_avg_so2_1h_df = join_avg_so2_1h_df.withColumn("min_dis", min(col("dis")).over(min_dis_window)).filter(col("min_dis") == col("dis"))
join_avg_so2_1h_df = join_avg_so2_1h_df.drop(col("dis"), col("min_dis"))

In [ ]:
join_avg_so2_1h_df = join_avg_so2_1h_df.withColumn("aqi_so2_1h", ((col("avg_so2_1h_df.avg_so2_1h") - col("bp_so2_df.BP_so2")) * (col("bp_so2_df2.I") - col("bp_so2_df.I")) / (col("bp_so2_df2.BP_so2") - col("bp_so2_df.BP_so2")) + col("bp_so2_df.I")))

In [ ]:
join_avg_so2_1h_df = join_avg_so2_1h_df.withColumn("date", col("time").substr(1, 10))

In [ ]:
max_aqi_window = Window.partitionBy(col("date"), col("city"), col("district")).orderBy(col("avg_so2_1h_df.avg_so2_1h").desc())

join_avg_so2_1h_df = join_avg_so2_1h_df.withColumn("max_avg_so2_1h", max(col("avg_so2_1h_df.avg_so2_1h")).over(max_aqi_window)) \
                    .filter(col("max_avg_so2_1h") == col("avg_so2_1h_df.avg_so2_1h")) \
                    .drop(col("max_avg_so2_1h"))

In [ ]:
join_avg_so2_1h_df.show(2)

+----------------+------+--------+----------+---+------+---+------+----------+----------+
|            time|  city|district|avg_so2_1h|  I|BP_so2|  I|BP_so2|aqi_so2_1h|      date|
+----------------+------+--------+----------+---+------+---+------+----------+----------+
|2022-08-04T18:00|Ha Noi| Ba Dinh|      22.9|  0|     0| 50|   125|      9.16|2022-08-04|
|2022-08-05T22:00|Ha Noi| Ba Dinh|      32.0|  0|     0| 50|   125|      12.8|2022-08-05|
+----------------+------+--------+----------+---+------+---+------+----------+----------+
only showing top 2 rows


#### AQI calculation for co

In [ ]:
join_avg_co_1h_df = avg_co_1h_df.alias("avg_co_1h_df").join(bp_co_df.alias("bp_co_df"), col("avg_co_1h_df.avg_co_1h") >= col("bp_co_df.BP_co"), how="left")
join_avg_co_1h_df = join_avg_co_1h_df.join(bp_co_df.alias("bp_co_df2"), col("avg_co_1h_df.avg_co_1h") < col("bp_co_df2.BP_co"), how="left")

In [ ]:
join_avg_co_1h_df = join_avg_co_1h_df.withColumn("dis", col("bp_co_df2.BP_co") - col("bp_co_df.BP_co"))
join_avg_co_1h_df = join_avg_co_1h_df.withColumn("min_dis", min(col("dis")).over(min_dis_window)).filter(col("min_dis") == col("dis"))
join_avg_co_1h_df = join_avg_co_1h_df.drop(col("dis"), col("min_dis"))

In [ ]:
join_avg_co_1h_df = join_avg_co_1h_df.withColumn("aqi_co_1h", (col("avg_co_1h_df.avg_co_1h") - col("bp_co_df.BP_co")) * (col("bp_co_df2.I") - col("bp_co_df.I")) / (col("bp_co_df2.BP_co") - col("bp_co_df.BP_co")) + col("bp_co_df.I"))

In [ ]:
join_avg_co_1h_df = join_avg_co_1h_df.withColumn("date", col("time").substr(1, 10))

In [ ]:
max_aqi_window = Window.partitionBy(col("date"), col("city"), col("district")).orderBy(col("avg_co_1h_df.avg_co_1h").desc())

join_avg_co_1h_df = join_avg_co_1h_df.withColumn("max_avg_co_1h", max(col("avg_co_1h_df.avg_co_1h")).over(max_aqi_window)) \
                    .filter(col("max_avg_co_1h") == col("avg_co_1h_df.avg_co_1h")) \
                    .drop(col("max_avg_co_1h"))

In [ ]:
join_avg_co_1h_df.show(2)

+----------------+------+--------+---------+---+-----+---+-----+---------+----------+
|            time|  city|district|avg_co_1h|  I|BP_co|  I|BP_co|aqi_co_1h|      date|
+----------------+------+--------+---------+---+-----+---+-----+---------+----------+
|2022-08-04T19:00|Ha Noi| Ba Dinh|    891.0|  0|    0| 50|10000|    4.455|2022-08-04|
|2022-08-05T19:00|Ha Noi| Ba Dinh|   1055.0|  0|    0| 50|10000|    5.275|2022-08-05|
+----------------+------+--------+---------+---+-----+---+-----+---------+----------+
only showing top 2 rows


#### AQI calculation for o3_1h

In [ ]:
join_avg_o3_1h_df = avg_o3_1h_df.alias("avg_o3_1h_df").join(bp_o3_1h_df.alias("bp_o3_1h_df"), col("avg_o3_1h_df.avg_o3_1h") >= col("bp_o3_1h_df.BP_o3_1h"), how="left")
join_avg_o3_1h_df = join_avg_o3_1h_df.join(bp_o3_1h_df.alias("bp_o3_1h_df2"), col("avg_o3_1h_df.avg_o3_1h") < col("bp_o3_1h_df2.BP_o3_1h"), how="left")

In [ ]:
join_avg_o3_1h_df = join_avg_o3_1h_df.withColumn("dis", col("bp_o3_1h_df2.BP_o3_1h") - col("bp_o3_1h_df.BP_o3_1h"))
join_avg_o3_1h_df = join_avg_o3_1h_df.withColumn("min_dis", min(col("dis")).over(min_dis_window)).filter(col("min_dis") == col("dis"))
join_avg_o3_1h_df = join_avg_o3_1h_df.drop(col("dis"), col("min_dis"))

In [ ]:
join_avg_o3_1h_df = join_avg_o3_1h_df.withColumn("aqi_o3_1h", (col("avg_o3_1h_df.avg_o3_1h") - col("bp_o3_1h_df.BP_o3_1h")) * (col("bp_o3_1h_df2.I") - col("bp_o3_1h_df.I")) / (col("bp_o3_1h_df2.BP_o3_1h") - col("bp_o3_1h_df.BP_o3_1h")) + col("bp_o3_1h_df.I"))

In [ ]:
join_avg_o3_1h_df = join_avg_o3_1h_df.withColumn("date", col("time").substr(1, 10))

In [ ]:
max_aqi_window = Window.partitionBy(col("date"), col("city"), col("district")).orderBy(col("avg_o3_1h_df.avg_o3_1h").desc())

join_avg_o3_1h_df = join_avg_o3_1h_df.withColumn("max_avg_o3_1h", max(col("avg_o3_1h_df.avg_o3_1h")).over(max_aqi_window)) \
                    .filter(col("max_avg_o3_1h") == col("avg_o3_1h_df.avg_o3_1h")) \
                    .drop(col("max_avg_o3_1h"))

In [ ]:
join_avg_o3_1h_df.show(2)

+----------------+------+--------+---------+---+--------+---+--------+---------+----------+
|            time|  city|district|avg_o3_1h|  I|BP_o3_1h|  I|BP_o3_1h|aqi_o3_1h|      date|
+----------------+------+--------+---------+---+--------+---+--------+---------+----------+
|2022-08-04T13:00|Ha Noi| Ba Dinh|    194.0| 50|     160|100|     200|     92.5|2022-08-04|
|2022-08-05T15:00|Ha Noi| Ba Dinh|    135.0|  0|       0| 50|     160|  42.1875|2022-08-05|
+----------------+------+--------+---------+---+--------+---+--------+---------+----------+
only showing top 2 rows


#### AQI calculation for o3_8h

In [ ]:
join_avg_o3_8h_df = avg_o3_8h_df.alias("avg_o3_8h_df").join(bp_o3_8h_df.alias("bp_o3_8h_df"), col("avg_o3_8h_df.avg_o3_8h") >= col("bp_o3_8h_df.BP_o3_8h"), how="left")
join_avg_o3_8h_df = join_avg_o3_8h_df.join(bp_o3_8h_df.alias("bp_o3_8h_df2"), col("avg_o3_8h_df.avg_o3_8h") < col("bp_o3_8h_df2.BP_o3_8h"), how="left")

In [ ]:
join_avg_o3_8h_df = join_avg_o3_8h_df.withColumn("dis", col("bp_o3_8h_df2.BP_o3_8h") - col("bp_o3_8h_df.BP_o3_8h"))
join_avg_o3_8h_df = join_avg_o3_8h_df.withColumn("min_dis", min(col("dis")).over(min_dis_window)).filter(col("min_dis") == col("dis"))
join_avg_o3_8h_df = join_avg_o3_8h_df.drop(col("dis"), col("min_dis"))

In [ ]:
join_avg_o3_8h_df = join_avg_o3_8h_df.withColumn("aqi_o3_8h", (col("avg_o3_8h_df.avg_o3_8h") - col("bp_o3_8h_df.BP_o3_8h")) * (col("bp_o3_8h_df2.I") - col("bp_o3_8h_df.I")) / (col("bp_o3_8h_df2.BP_o3_8h") - col("bp_o3_8h_df.BP_o3_8h")) + col("bp_o3_8h_df.I"))

In [ ]:
join_avg_o3_8h_df = join_avg_o3_8h_df.withColumn("date", col("time").substr(1, 10))

In [ ]:
max_aqi_window = Window.partitionBy(col("date"), col("city"), col("district")).orderBy(col("avg_o3_8h_df.avg_o3_8h").desc())

join_avg_o3_8h_df = join_avg_o3_8h_df.withColumn("max_avg_o3_8h", max(col("avg_o3_8h_df.avg_o3_8h")).over(max_aqi_window)) \
                    .filter(col("max_avg_o3_8h") == col("avg_o3_8h_df.avg_o3_8h")) \
                    .drop(col("max_avg_o3_8h"))

In [ ]:
join_avg_o3_8h_df.show(2)

+----------------+------+--------+---------+---+--------+---+--------+---------+----------+
|            time|  city|district|avg_o3_8h|  I|BP_o3_8h|  I|BP_o3_8h|aqi_o3_8h|      date|
+----------------+------+--------+---------+---+--------+---+--------+---------+----------+
|2022-08-04T17:00|Ha Noi| Ba Dinh|  160.125|100|   120.0|150|   170.0|  140.125|2022-08-04|
|2022-08-05T17:00|Ha Noi| Ba Dinh|    114.5| 50|   100.0|100|   120.0|    86.25|2022-08-05|
+----------------+------+--------+---------+---+--------+---+--------+---------+----------+
only showing top 2 rows


In [ ]:
print(join_avg_pm25_24h_df.count())
print(join_avg_pm10_24h_df.count())
print(join_avg_co_1h_df.count())
print(join_avg_so2_1h_df.count())
print(join_avg_no2_1h_df.count())
print(join_avg_o3_1h_df.count())
print(join_avg_o3_8h_df.count())

43
42
44
44
42
46
43


In [ ]:
final_aqi_df = join_avg_pm10_24h_df.join(join_avg_pm25_24h_df, on=["date", "city", "district"], how="inner") \
                    .join(join_avg_co_1h_df, on=["date", "city", "district"], how="inner") \
                    .join(join_avg_so2_1h_df, on=["date", "city", "district"], how="inner") \
                    .join(join_avg_no2_1h_df, on=["date", "city", "district"], how="inner") \
                    .join(join_avg_o3_1h_df, on=["date", "city", "district"], how="inner") \
                    .join(join_avg_o3_8h_df, on=["date", "city", "district"], how="inner") \
                    .select("date", "city", "district", "aqi_pm25_24h", "aqi_pm10_24h", "aqi_co_1h", "aqi_so2_1h", "aqi_no2_1h", "aqi_o3_1h", "aqi_o3_8h")

In [ ]:
final_aqi_df = final_aqi_df.withColumn("AQI", greatest(col("aqi_pm25_24h"), col("aqi_pm10_24h"), col("aqi_co_1h"), col("aqi_so2_1h"), col("aqi_no2_1h"), col("aqi_o3_1h"), col("aqi_o3_8h")))

In [ ]:
final_aqi_df.count()

52

In [ ]:
final_aqi_df.select(min(col("AQI")), max(col("AQI")), avg(col("AQI"))).show()

+--------+------------------+------------------+
|min(AQI)|          max(AQI)|          avg(AQI)|
+--------+------------------+------------------+
|  54.375|202.82894736842104|115.84952987436539|
+--------+------------------+------------------+



| Khoảng giá trị AQI | Chất lượng không khí | Màu sắc   | Mã màu RGB   |
|--------------------|----------------------|-----------|--------------|
| 0 - 50             | Tốt                  | Xanh      | 0;228;0      |
| 51 - 100           | Trung bình           | Vàng      | 255;255;0    |
| 101 - 150          | Kém                  | Da cam    | 255;126;0    |
| 151 - 200          | Xấu                  | Đỏ        | 255;0;0      |
| 201 - 300          | Rất xấu              | Tím       | 143;63;151   |
| 301 - 500          | Nguy hại             | Nâu       | 126;0;35     |
